In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install --no-cache-dir "bitsandbytes>=0.43.1" "transformers>=4.44.2"

import torch, sys
print("Torch:", torch.__version__)
# GPU check
if torch.cuda.is_available():
    ngpu = torch.cuda.device_count()
    caps = [torch.cuda.get_device_capability(i) for i in range(ngpu)]
    print("CUDA GPUs:", ngpu, "capabilities:", caps)
else:
    print("No CUDA GPU detected. 4-bit will not work.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 218.1 MB/s eta 0:00:00
Torch: 2.8.0+cu126
CUDA GPUs: 1 capabilities: [(7, 5)]


In [ ]:
from pathlib import Path

# mounting
CANDIDATE_ROOTS = [Path("/content/drive/MyDrive"), Path("/content/drive/My Drive")]
DRIVE_ROOT = next((p for p in CANDIDATE_ROOTS if p.exists()), None)

PROJECT_DIR  = DRIVE_ROOT / "CUIT"
ADAPTER_DIR  = PROJECT_DIR / "qwen3_lora_adapter_v3" ############################ HERE CHANGE TO model
TEST_PATH = PROJECT_DIR / "test_real.jsonl"
OUT_JSONL = PROJECT_DIR / "predictions_det3.jsonl" ############################ HERE CHANGE TO OUT PUT
OUT_CSV   = PROJECT_DIR / "predictions_det3.csv" ############################ HERE CHANGE TO OUT PUT

print("Adapter :", ADAPTER_DIR)
print("Test    :", TEST_PATH)

Adapter : /content/drive/MyDrive/CUIT/qwen3_lora_adapter_v3
Test    : /content/drive/MyDrive/CUIT/test_real.jsonl


In [ ]:
import json, torch, math
from tqdm import tqdm
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

import re
from typing import List, Dict
import torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

can_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if can_bf16 else torch.float16

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    quantization_config=bnb_cfg,
    torch_dtype=compute_dtype,
)
base.config.use_cache = True

model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval() # use base for raw qwen


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
def load_test_real_jsonl(path) -> List[Dict]:
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            msgs = rec.get("messages", [])
            gold = ""
            if msgs and msgs[-1].get("role") == "assistant":
                gold = msgs[-1].get("content", "") or ""
                msgs_in = msgs[:-1] # strip gold
            else:
                msgs_in = msgs
            items.append({"messages_in": msgs_in, "gold": gold, "raw": rec})
    return items

test_items = load_test_real_jsonl(TEST_PATH)
print("Loaded items:", len(test_items))

Loaded items: 15


In [ ]:
@torch.inference_mode()
def generate_one(messages_in,
                 max_new_tokens=512,
                 temperature=0.6,
                 top_p=0.95,
                 do_sample=False):
    input_ids = tokenizer.apply_chat_template(
        messages_in,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    out = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=top_p,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    gen_ids = out[0, input_ids.shape[1]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return text.strip()


In [ ]:
from tqdm import tqdm
import pandas as pd

rows = []
with open(OUT_JSONL, "w", encoding="utf-8") as fout:
    for i, item in enumerate(tqdm(test_items, desc="Generating")):
        msgs_in = item["messages_in"]
        gold    = item["gold"]

        # inference
        pred = generate_one(msgs_in)

        # last user text
        user_txt = ""
        for m in reversed(msgs_in):
            if m.get("role") == "user":
                user_txt = m.get("content", "")
                break

        rec_out = {
            "index": i,
            "user": user_txt,
            "gold": gold,
            "prediction": pred,
            "messages_in": msgs_in,  # input given to model
        }
        fout.write(json.dumps(rec_out, ensure_ascii=False) + "\n")
        rows.append({"index": i, "user": user_txt, "gold": gold, "prediction": pred})

pd.DataFrame(rows).to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUT_JSONL)
print("Saved:", OUT_CSV)

Generating: 100%|██████████| 15/15 [06:50<00:00, 27.35s/it]

Saved: /content/drive/MyDrive/CUIT/predictions_det3.jsonl
Saved: /content/drive/MyDrive/CUIT/predictions_det3.csv
